In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch as tn
from igakit import cad

from ttnte.xs.benchmarks import c5g7
from ttnte.iga import IGAMesh
from ttnte.assemblers import MatrixAssembler, TTAssembler
from ttnte.linalg import LinearOperator, eig

tn.set_default_dtype(tn.float64)
tn.set_num_threads(14)

In [ ]:
# Discretization
num_ordinates = 256

# Get XS data
xs_server = c5g7()

In [ ]:
# Create quarter circle NURBS surface
radius = 0.54 # cm
pitch = 1.26 # cm
c0 = cad.circle(radius=radius, angle=np.pi / 2)
c1 = c0.slice(0, 0, 0.5)
c2 = c0.slice(0, 0.5, 1)
l0 = cad.line(p0=(0, 0), p1=(0, 0))
fuel1 = cad.ruled(l0, c1)
fuel2 = cad.ruled(l0, c2)

# Create water patch
l1 = cad.line(p0=(pitch / 2, 0), p1=(pitch / 2, pitch / 2))
l2 = cad.line(p0=(pitch / 2, pitch / 2), p1=(0, pitch / 2))
mod1 = cad.ruled(c1, l1)
mod2 = cad.ruled(c2, l2)

# NURBS surfaces
patches = {}
patches[fuel1] = "UO2"
patches[fuel2] = "UO2"
patches[mod1] = "Water"
patches[mod2] = "Water"

# Create IGA mesh object
mesh = IGAMesh(patches)

# Refine mesh
for p in range(mesh.num_patches):
    mesh.refine(p, 6, 2)

# Finalize mesh
mesh.connect()

# Set reflective boundary conditions
mesh.set_reflective_condition(("left", "bottom", "top", "right"))

# Finalize mesh
mesh.finalize()

In [ ]:
# Plot final mesh
ax = mesh.plot()
ax.view_init(90, -90, 0)
plt.legend()
plt.tight_layout()
plt.savefig("./figs/pincell.png", dpi=300)
plt.show()

In [ ]:
# Create operators in TT format
assembler = TTAssembler(
    mesh=mesh,
    xs_server=xs_server,
    num_ordinates=num_ordinates,
)
assembler.interp_jacobian = False
assembler.interp_jacobian_det = False
assembler.interp_boundary_jacobian_det = False
H, S, F, B_in, B_out = assembler.build(use_tt=True, eps=1e-10)

# Save TT information
assembler.save_tt_info("./tt_info.csv")

In [ ]:
k, psi = eig(
    LHS=LinearOperator([H, B_out, -S, -B_in], N=assembler.N, M=assembler.M),
    RHS=LinearOperator([F], N=assembler.N, M=assembler.M),
    tols=1e-8,
    max_iters=500,
    device=0,
)

# Compute scalar flux
phi = assembler.angular_integral(psi).numpy().reshape(
    (xs_server.num_groups, mesh.num_patches, -1)
)

# Calculate eigenvalue error
print("keff error: {} pcm".format((k - 1) * 1e5))

In [ ]:
# Iterate through groups and plot
for g in range(xs_server.num_groups):
    # Set control points
    mesh.set_phi(phi[g,])

    # Plot
    plt.clf()
    fig = mesh.plot(backend="plotly")
    fig.update_layout(scene={"zaxis_title": f"Scalar Flux (g = {g + 1})"})
    fig.write_html(f"./figs/phi_{g + 1}.html")